In [1]:
!pip install -U langchain langchain-core langchain-community langchain-openai openai langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.30
    Uninstalling langsmith-0.7.30:
      Successfully uninstalled langsmith-0.7.30
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain

In [2]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE_YOUR_OPENAI_KEY"
os.environ["LANGCHAIN_API_KEY"] = "lsv2_pt_b7146c0d05f24f83971ad97bd8d85047_33caxxxxxx"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [5]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract skills, tools, and experience from this resume.
Return JSON.

Resume:
{resume}
"""
)

match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
Compare resume with job description.

Return:
- matched skills
- missing skills

Resume Data:
{resume_data}

Job:
{job_description}
"""
)

score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give a score from 0 to 100 based on matching.

Match Data:
{match_data}
"""
)

explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template="""
Explain why this score was given.

Score: {score}
Match Data: {match_data}
"""
)

In [6]:
extract_chain = extract_prompt | llm | StrOutputParser()
match_chain = match_prompt | llm | StrOutputParser()
score_chain = score_prompt | llm | StrOutputParser()
explain_chain = explain_prompt | llm | StrOutputParser()

In [7]:
def run_pipeline(resume, job):
    extracted = extract_chain.invoke({"resume": resume})

    matched = match_chain.invoke({
        "resume_data": extracted,
        "job_description": job
    })

    score = score_chain.invoke({"match_data": matched})

    explanation = explain_chain.invoke({
        "score": score,
        "match_data": matched
    })

    print("----- EXTRACTED -----")
    print(extracted)

    print("\n----- MATCHED -----")
    print(matched)

    print("\n----- SCORE -----")
    print(score)

    print("\n----- EXPLANATION -----")
    print(explanation)

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [12]:
!pip install transformers

In [13]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
def run_pipeline(resume, job):
    e = generator(f"Extract skills from: {resume}", max_length=150)[0]['generated_text']
    m = generator(f"Match resume {e} with job {job}", max_length=150)[0]['generated_text']
    s = generator(f"Give score for: {m}", max_length=100)[0]['generated_text']
    ex = generator(f"Explain score: {s}", max_length=150)[0]['generated_text']

    print("EXTRACTED:", e)
    print("\nMATCH:", m)
    print("\nSCORE:", s)
    print("\nEXPLANATION:", ex)

In [15]:
run_pipeline(strong_resume, job)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_l

EXTRACTED: Extract skills from: 
Python, Machine Learning, Deep Learning, NLP,
TensorFlow, PyTorch, 3 years experience

Bachelor of Science in Computer Science from University of Colorado, Boulder, or

Bachelor of Science in Mathematics from University of Colorado, Boulder, or

Bachelor of Science in English from University of Colorado, Boulder, or

Bachelor of Science in Psychology from University of Colorado, Boulder, or

Bachelor of Science in English from University of Colorado, Boulder, or

Bachelor of Science in Philosophy from University of Colorado, Boulder, or

MATCH: Match resume Extract skills from: 
Python, Machine Learning, Deep Learning, NLP,
TensorFlow, PyTorch, 3 years experience

Bachelor of Science in Computer Science from University of Colorado, Boulder, or

Bachelor of Science in Mathematics from University of Colorado, Boulder, or

Bachelor of Science in English from University of Colorado, Boulder, or

Bachelor of Science in Psychology from University of Colorado,

In [16]:
average_resume = "Python, Pandas, basic ML, 1 year experience"
weak_resume = "Excel, MS Word, no programming"

In [17]:
run_pipeline(strong_resume, job)
run_pipeline(average_resume, job)
run_pipeline(weak_resume, job)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

EXTRACTED: Extract skills from: 
Python, Machine Learning, Deep Learning, NLP,
TensorFlow, PyTorch, 3 years experience

Introduction to Python

Introduction to Python is a series of books on Python that aim to help you understand what it is like to code. These books have been written for people who are not familiar with Python, but who have learned to code Python using Python in other languages.

The book covers Python basics, including the implementation of functions, functions, and functions with standard Python functions. While Python basics are a great read for programmers new to Python, there are many tutorials that are based on the standard Python tutorials.

This book will allow you to learn Python on your own; use it to write applications, test out programming languages, and learn new features and techniques.

How to learn Python

Learn to code with Python, but not Python 3

Learn Python with Python 3

Learn Python, but not Python 3

Learn Python, but not Python 3

Learn Python

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

EXTRACTED: Extract skills from: Python, Pandas, basic ML, 1 year experience, Python 3/4, 3 years experience in web development

Solve problems: Machine learning, machine learning, machine learning with a single problem

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full time (40 hours)

Apply to: Full

MATCH: Match resume Extract skills from: Python, Pandas, basic ML, 1 year experience, Python 3/4, 3 years experience in web 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXTRACTED: Extract skills from: Excel, MS Word, no programming, Excel, Wordpress, Pivot Table, Wordpress 2, Excel 2003, Excel 2010, Excel 2011, Excel 2012, Excel 2013, Excel 2014, and Excel 2015. If you use your own Excel skills in the above example you will need to add more skills to your skills list.

Step 3. Write down your skills in Word and Excel

Step 3.1. Write down your skills in Word and Excel

Step 3.2. Write down your skills in Word and Excel

Step 3.3. Copy and paste the training document to your Word and Excel

Step 3.4. If you want to use the Training Document, add a line to the end of your training document.

Step 3.5. Copy and Paste the Training Document to your Word and Excel

Step 3.6. If you want to use the Training Document, add a line to the end of your training document.

Step 3.7. If you want to use the Training Document, add a line to the end of your training document.

Step 3.8. If you want to use the Training Document, add a line to the end of your training do